# WSE Hawkes Calibration

Fit a 6-dimensional Hawkes process to WSE order flow with the
`HawkesCalibration` class in `research_core.classes`. The production model is
the single-exponential kernel; every figure in the thesis comes from it.

## Kernel

Each ordered pair of event types $(i, j)$ gets one exponential kernel
$\phi_{ij}(t) = \alpha_{ij}\,\beta_{ij}\,e^{-\beta_{ij} t}$. Optuna searches the
decay rates $\beta_{ij}$; `tick` then returns the baseline $\mu_i$ and the
adjacency $\alpha_{ij}$ by maximum likelihood.

## Event dimensions

`MO_bid`, `MO_ask`, `LO_bid`, `LO_ask`, `CXL_bid`, `CXL_ask`. By WSE convention
`MO_bid` is an aggressive buy and `MO_ask` an aggressive sell (see
`data/schema.py`).

## Steps

1. Poisson baseline, in raw clock time and seasonality-adjusted $\tau$-time.
2. Univariate single-exponential Hawkes, one fit per dimension.
3. Multivariate single-exponential Hawkes, the production calibration.
4. Save the fitted parameters.

The sum-of-exponential ("triple kernel") experiment and the per-quintet runs
sit in the appendix at the bottom. Nothing above depends on them, so the whole
appendix can be removed in one block if the thesis stays single-exponential.

In [ ]:
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from research_core.classes import (
    HawkesCalibration,
    helpers,
    compute_end_times,
    list_day_keys,
    load_events_from_sqlite_bulk,
    estimate_seasonality_profiles,
    project_root,
    data_dir,
    run_full_extraction
)
from research_core.classes.calibrate import (
    get_average_seasonality_shape,
    plot_all_seasonality_patterns,
)

In [ ]:
ROOT = project_root()

ASSETS = {"KGHM": 10783, "PKNORLEN": 11319, "PKOBP": 11314, "PZU": 10735, "PEKAO": 11322}

asset = "KGHM"

# Evaluation window: skip the first DAY_START_IDX sessions (warm-up / illiquid
# listing days), then take N_DAYS. Extraction, the bulk loader, and the quintet
# script all use this same window.
N_DAYS = 229
DAY_START_IDX = 6

MARKET_OPEN = "09:00:00"

marks_order = ["MO_bid", "MO_ask", "LO_bid", "LO_ask", "CXL_bid", "CXL_ask"]

MAX_ITER = 1_000_000
TOL = 1e-9

GOF_DIMS = ["LO_bid", "MO_bid"]

DATA_DIR = data_dir()

In [ ]:
FORCE_REEXTRACT = False
MARKET_CLOSE = "16:50:00"
TICK_SIZE = 0.05

db_path = DATA_DIR / f"{asset}_order_flow.sqlite"
orders_h5 = ROOT / "data" / "WSE data" / "WSELOB-2017" / "orders" / f"{asset}_lob_2017_zlib.h5"
trades_h5 = ROOT / "data" / "WSE data" / "WSELOB-2017" / "trades" / f"{asset}_trades_2017_zlib.h5"

all_days = list_day_keys(asset)
selected_days = all_days[DAY_START_IDX : DAY_START_IDX + N_DAYS]
print(f"Day window: {selected_days[0]} .. {selected_days[-1]} ({len(selected_days)} days)")

if FORCE_REEXTRACT or not db_path.exists():
    run_full_extraction(
        asset, orders_h5, trades_h5, db_path,
        market_open=MARKET_OPEN,
        market_close=MARKET_CLOSE,
        tick_size=TICK_SIZE,
        force=FORCE_REEXTRACT,
        day_keys=selected_days,
    )
else:
    print(f"SQLite DB already exists: {db_path}  (set FORCE_REEXTRACT=True to rebuild)")

In [ ]:
# Same window as the extraction cell above. Loading all_days[:N_DAYS] instead
# would skip the DAY_START_IDX offset and silently calibrate on the wrong days.
day_keys = list_day_keys(asset)[DAY_START_IDX : DAY_START_IDX + N_DAYS]
print(f"Selected {len(day_keys)} days: {day_keys[0]} .. {day_keys[-1]}")

print(f"\nBulk-loading events from SQLite ({len(day_keys)} days)...")
t0 = time.time()
timestamps_by_day = load_events_from_sqlite_bulk(
    db_path, day_keys, MARKET_OPEN, marks_order,
)
print(f"Bulk load complete in {time.time() - t0:.2f}s")

for dk, day_seq in [(day_keys[0], timestamps_by_day[0]),
                     (day_keys[-1], timestamps_by_day[-1])]:
    counts = {m: len(s) for m, s in zip(marks_order, day_seq)}
    max_t = max((s.max() if len(s) else 0.0) for s in day_seq)
    print(f"  {dk}: {counts}  total={sum(counts.values())}  max_t={max_t:.2f}")

end_times = compute_end_times(timestamps_by_day)
print(f"\nDays: {len(timestamps_by_day)}, max end time: {end_times.max():.2f} s")

# Intraday Seasonality Estimation

We estimate a nonparametric intraday intensity profile $s^{(k)}(t)$ for each event type using an Epanechnikov kernel,
normalised so that $\frac{1}{T}\int_0^T s^{(k)}(t)\,dt = 1$.

The average shape across all event types is then used to construct a shared time transformation
$\tau(t) = \int_0^t \bar{s}(u)\,du$ that removes deterministic clock-time variation.

In [ ]:
SEASONALITY_BANDWIDTH = 300.0
SEASONALITY_GRID_POINTS = 400
FORCE_SEASONALITY_RECOMPUTE = False

seasonality_cache_dir = DATA_DIR / "seasonality_cache"
seasonality_cache_path = seasonality_cache_dir / f"{asset}_seasonality_profiles.pkl"

# Use ALL available days for seasonality estimation (not just the calibration subset)
all_day_keys_season = list_day_keys(asset)
print(f"Loading ALL {len(all_day_keys_season)} days for seasonality estimation...")

t0 = time.time()
all_ts_season = load_events_from_sqlite_bulk(
    db_path, all_day_keys_season, MARKET_OPEN, marks_order,
)
all_end_times_season = compute_end_times(all_ts_season)
print(f"Loaded in {time.time() - t0:.2f}s")

seasonality_profiles = estimate_seasonality_profiles(
    all_ts_season,
    marks_order,
    all_end_times_season,
    bandwidth=SEASONALITY_BANDWIDTH,
    grid_points=SEASONALITY_GRID_POINTS,
    cache_path=seasonality_cache_path,
    force_recompute=FORCE_SEASONALITY_RECOMPUTE,
)

In [ ]:
fig, ax, seasonal_stats = plot_all_seasonality_patterns(
    seasonality_profiles,
    marks_order=marks_order,
    normalize=True,
    show_average=True,
    show_uncertainty=True,
)
plt.show()

avg_seasonality = get_average_seasonality_shape(
    seasonality_profiles,
    marks_order=marks_order,
    normalize=True,
)
print(f"Average seasonality grid points: {len(avg_seasonality['grid'])}")
print(f"Total tau at end of day: {avg_seasonality['cumulative_integral'][-1]:.4f}")

## Instantiate HawkesCalibration

All subsequent calibrations are run through a single `HawkesCalibration` instance.
By passing `seasonality_profiles`, the class automatically builds the shared average
$\tau$-time transformation internally. Methods accept `use_tau=True/False`.

In [ ]:
cal = HawkesCalibration(
    timestamps_by_day=timestamps_by_day,
    marks_order=marks_order,
    end_times=end_times,
    seasonality_profiles=seasonality_profiles,
    max_iter=MAX_ITER,
    tol=TOL
)

print("HawkesCalibration ready:")
print(f"  Dimensions:   {cal.n_nodes}")
print(f"  Days:         {cal.n_days}")
print(f"  Marks:        {cal.marks_order}")
print(f"  Seasonality:  {'yes' if cal.seasonality_profiles else 'no'}")
print(f"  Fixed betas:  fast={cal.BETA_FAST}, slow={cal.BETA_SLOW}")

## 1. Poisson baseline

Constant-rate MLE: $\hat\mu_i = N_i / T_{\text{total}}$.

Fitted in both raw clock time and seasonality-adjusted $\tau$-time.

In [ ]:
poisson_raw = cal.fit_poisson(
    use_tau=False,
    day_keys=day_keys,
    gof_dims=GOF_DIMS,
)

In [ ]:
poisson_tau = cal.fit_poisson(
    use_tau=True,
    day_keys=day_keys,
    gof_dims=GOF_DIMS,
)

## 2. Univariate Hawkes -- single exponential

Each dimension is fitted independently with a single-exponential kernel:
$\phi_{ii}(t) = \alpha_{ii}\beta_{ii}e^{-\beta_{ii}t}$, with $\beta_{ii}$ searched via Optuna.

In [ ]:
uni_single_raw = cal.fit_univariate_hawkes(
    use_tau=False,
    n_trials=100,
    beta_min=0.01,
    beta_max=20.0,
    gof_dims=GOF_DIMS,
)

In [ ]:
uni_single_tau = cal.fit_univariate_hawkes(
    use_tau=True,
    n_trials=100,
    beta_min=0.01,
    beta_max=20.0,
    gof_dims=GOF_DIMS,
)

## 3. Multivariate Hawkes -- single exponential

Full 6x6 mutually-exciting Hawkes with per-kernel decay $\beta_{ij}$ searched
via parallel Optuna (subprocess workers).

In [ ]:
multi_single_raw = cal.fit_multivariate_hawkes(
    use_tau=False,
    n_trials=1000,
    n_workers=12,
    beta_min=0.1,
    beta_max=20.0,
    gof_dims=GOF_DIMS,
)

n_nodes = cal.n_nodes

plt.figure(figsize=(6, 5))
plt.imshow(multi_single_raw["adjacency"], cmap="viridis")
plt.colorbar(label="alpha")
plt.xticks(range(n_nodes), marks_order, rotation=45)
plt.yticks(range(n_nodes), marks_order)
plt.title(r"Adjacency matrix $\alpha$ (raw time)")
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 5))
plt.imshow(multi_single_raw["decays"], cmap="viridis")
plt.colorbar(label="beta")
plt.xticks(range(n_nodes), marks_order, rotation=45)
plt.yticks(range(n_nodes), marks_order)
plt.title(r"Decay matrix $\beta$ (raw time)")
plt.tight_layout()
plt.show()

In [ ]:
multi_single_tau = cal.fit_multivariate_hawkes(
    use_tau=True,
    n_trials=1000,
    n_workers=12,
    beta_min=0.1,
    beta_max=20.0,
    gof_dims=GOF_DIMS,
)

plt.figure(figsize=(6, 5))
plt.imshow(multi_single_tau["adjacency"], cmap="viridis")
plt.colorbar(label="alpha")
plt.xticks(range(n_nodes), marks_order, rotation=45)
plt.yticks(range(n_nodes), marks_order)
plt.title(r"Adjacency matrix $\alpha$ ($\tau$-time)")
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 5))
plt.imshow(multi_single_tau["decays"], cmap="viridis")
plt.colorbar(label="beta")
plt.xticks(range(n_nodes), marks_order, rotation=45)
plt.yticks(range(n_nodes), marks_order)
plt.title(r"Decay matrix $\beta$ ($\tau$-time)")
plt.tight_layout()
plt.show()

## Save calibrated parameters

All fitted parameter sets are saved to pickle files for downstream use
(simulation, analysis, etc.).

In [ ]:
HawkesCalibration.save_params(
    f"multivariate_hawkes_params_{asset}.pkl",
    marks=marks_order,
    baseline=multi_single_tau["baseline"],
    adjacency=multi_single_tau["adjacency"],
    decays=multi_single_tau["decays"],
)

core_results = {
    "asset": asset,
    "marks_order": marks_order,
    "n_days": len(day_keys),
    "day_keys": day_keys,
    "poisson_raw": poisson_raw,
    "poisson_tau": poisson_tau,
    "uni_single_raw": {k: v for k, v in uni_single_raw.items() if k != "models"},
    "uni_single_tau": {k: v for k, v in uni_single_tau.items() if k != "models"},
    "multi_single_raw": {k: v for k, v in multi_single_raw.items() if k != "model"},
    "multi_single_tau": {k: v for k, v in multi_single_tau.items() if k != "model"},
}

HawkesCalibration.save_params(
    f"hawkes_calibration_full_{asset}.pkl",
    **core_results,
)

print("\nSingle-exponential calibration saved.")

# Appendix: sum-of-exponential (triple) kernels and quintets

Everything below is experimental and not part of the production model. It fits
sum-of-exponential kernels, runs the per-quintet calibration, and validates the
quintet simulations. Nothing in the core path above reads any of it, so the
whole appendix can be deleted in one go if the thesis stays single-exponential.

The triple-kernel decay search needs `BETA_RANGES` and `SLOW_SELF_FLOOR`,
defined in the next cell. They are kept out of the main config on purpose.

## Sum-of-exponentials with Optuna decay search

Each kernel becomes a sum of three exponentials at fast, mid, and slow
timescales. Optuna searches the three shared decay rates; for each trial
`HawkesSumExpKern` refits the alpha matrices and baselines at the candidate
decays. The search bands come from `BETA_RANGES` below.

In [ ]:
# Decay-rate search bands for the sum-of-exponential kernels: one (min, max)
# band per timescale (fast, mid, slow). SLOW_SELF_FLOOR forces a minimum share
# of self-excitation onto the slow component for the two MO dimensions.
BETA_RANGES = [(50.0, 100.0), (1.0, 10.0), (0.001, 0.01)]
SLOW_SELF_FLOOR = {"dims": [0, 1], "r_target": 0.20}

In [ ]:
uni_sumexp_optuna_raw = cal.fit_univariate_hawkes_sumexp_optuna(
    use_tau=False,
    n_trials=200,
    n_workers=12,
    beta_ranges=BETA_RANGES,
    gof_dims=GOF_DIMS,
    penalty="l2",
    C=1e3,
)

In [ ]:
uni_sumexp_optuna_tau = cal.fit_univariate_hawkes_sumexp_optuna(
    use_tau=True,
    n_trials=200,
    n_workers=12,
    beta_ranges=BETA_RANGES,
    gof_dims=GOF_DIMS,
    penalty="l2",
    C=1e3,
)

In [ ]:
multi_sumexp_optuna_raw = cal.fit_multivariate_hawkes_sumexp_optuna(
    use_tau=False,
    n_trials=300,
    n_workers=12,
    beta_ranges=BETA_RANGES,
    gof_dims=GOF_DIMS,
    penalty="l2",
    C=1e3,
    slow_self_floor=SLOW_SELF_FLOOR,
)

In [ ]:
multi_sumexp_optuna_tau = cal.fit_multivariate_hawkes_sumexp_optuna(
    use_tau=True,
    n_trials=500,
    n_workers=12,
    beta_ranges=BETA_RANGES,
    gof_dims=GOF_DIMS,
    penalty="l2",
    C=1e3,
    slow_self_floor=SLOW_SELF_FLOOR,
)

In [ ]:
HawkesCalibration.save_params(
    f"hawkes_sumexp_params_{asset}.pkl",
    asset=asset,
    marks_order=marks_order,
    n_days=len(day_keys),
    day_keys=day_keys,
    uni_sumexp_optuna_raw={k: v for k, v in uni_sumexp_optuna_raw.items() if k != "models"},
    uni_sumexp_optuna_tau={k: v for k, v in uni_sumexp_optuna_tau.items() if k != "models"},
    multi_sumexp_optuna_raw={k: v for k, v in multi_sumexp_optuna_raw.items() if k != "model"},
    multi_sumexp_optuna_tau={k: v for k, v in multi_sumexp_optuna_tau.items() if k != "model"},
)

print("\nSum-of-exponential calibration saved (experimental).")

## Per-quintet triple Hawkes (100 calibration groups)

Requires **earlier cells** in this notebook: `timestamps_by_day`, `day_keys`, `end_times`, `marks_order`, `seasonality_profiles`, `MAX_ITER`, `TOL`, `BETA_RANGES` (length **3** = triple kernel), `GOF_DIMS`, `SLOW_SELF_FLOOR`, `asset`, `DATA_DIR`, and `HawkesCalibration`.

Loads `calibration_quintets.json` (same `day_keys` order as bulk load), slices **5 trading days** per quintet row, fits `fit_multivariate_hawkes_sumexp_optuna` in **τ-time** per group, and writes:

- `hawkes_triple_quintet_{asset}_{b:03d}.pkl`: parameters (no tick `model` object)
- `hawkes_triple_quintet_index_{asset}.pkl`: manifest of runs

Set `QUINTET_GOF_DIMS = []` below to skip GOF plots during the loop (default); set to `GOF_DIMS` if you want plots per quintet (slow).


In [ ]:
import json
from pathlib import Path

import numpy as np

# --- knobs ---
QUINTET_JSON = DATA_DIR / "calibration_quintets.json"
QUINTET_START = 0
QUINTET_END = 100  # exclusive; use 1 for a smoke test
QUINTET_N_TRIALS = 200
QUINTET_N_WORKERS = 12
SKIP_EXISTING = True
# GOF plots per quintet are expensive; keep empty unless debugging
QUINTET_GOF_DIMS: list = []

assert len(BETA_RANGES) == 3, "BETA_RANGES must have length 3 (triple sum-exp)"

with open(QUINTET_JSON, "r", encoding="utf-8") as fp:
    qmeta = json.load(fp)

quintet_rows = qmeta.get("quintets")
if not quintet_rows or len(quintet_rows) != 100:
    raise ValueError("calibration_quintets.json: expected key 'quintets' with 100 rows")

day_index = {dk: i for i, dk in enumerate(day_keys)}
index_entries: list = []

for b in range(QUINTET_START, min(QUINTET_END, 100)):
    row_days = [str(x) for x in quintet_rows[b]]
    out_path = DATA_DIR / f"hawkes_triple_quintet_{asset}_{b:03d}.pkl"
    if SKIP_EXISTING and out_path.is_file():
        print(f"[{b:03d}] skip (exists) {out_path.name}")
        try:
            prev = HawkesCalibration.load_params(out_path)
            betas = prev.get("best_betas")
            index_entries.append(
                {
                    "quintet": int(b),
                    "path": str(out_path.resolve()),
                    "score": float(prev.get("score", float("nan"))),
                    "branching_ratio": float(prev["branching_ratio"]),
                    "best_betas": np.asarray(betas).tolist() if betas is not None else None,
                    "quintet_day_keys": row_days,
                    "skipped": True,
                }
            )
        except Exception as e:
            print(f"[{b:03d}] skip: could not read pickle for index: {e}")
        continue

    try:
        idxs = [day_index[dk] for dk in row_days]
    except KeyError as e:
        msg = f"quintet {b}: day key missing from notebook day_keys: {e}"
        print(msg)
        index_entries.append({"quintet": b, "error": msg, "path": None})
        continue

    events_b = [timestamps_by_day[i] for i in idxs]
    end_b = end_times[np.asarray(idxs, dtype=int)]

    try:
        cal_b = HawkesCalibration(
            events_b,
            marks_order,
            end_b,
            seasonality_profiles=seasonality_profiles,
            max_iter=MAX_ITER,
            tol=TOL,
        )
        out_b = cal_b.fit_multivariate_hawkes_sumexp_optuna(
            use_tau=True,
            n_trials=QUINTET_N_TRIALS,
            n_workers=QUINTET_N_WORKERS,
            beta_ranges=BETA_RANGES,
            gof_dims=QUINTET_GOF_DIMS,
            penalty="l2",
            C=1e3,
            slow_self_floor=SLOW_SELF_FLOOR,
        )

        payload = {
            "marks_order": list(marks_order),
            "quintet_index": int(b),
            "quintet_day_keys": row_days,
            **{k: v for k, v in out_b.items() if k != "model"},
        }
        HawkesCalibration.save_params(out_path, **payload)

        betas = out_b.get("best_betas")
        index_entries.append(
            {
                "quintet": int(b),
                "path": str(out_path.resolve()),
                "score": float(out_b["score"]),
                "branching_ratio": float(out_b["branching_ratio"]),
                "best_betas": np.asarray(betas).tolist() if betas is not None else None,
                "quintet_day_keys": row_days,
            }
        )
        print(f"[{b:03d}] saved  score={out_b['score']:.4f}  rho={out_b['branching_ratio']:.4f}")
    except Exception as e:
        import traceback

        print(f"[{b:03d}] FAILED: {e}")
        traceback.print_exc()
        index_entries.append({"quintet": int(b), "error": repr(e), "path": None})

index_path = DATA_DIR / f"hawkes_triple_quintet_index_{asset}.pkl"
HawkesCalibration.save_params(
    index_path,
    asset=asset,
    quintet_json=str(QUINTET_JSON.resolve()),
    beta_ranges=BETA_RANGES,
    n_trials=QUINTET_N_TRIALS,
    entries=index_entries,
)
print("Wrote index:", index_path)


In [ ]:
# Optional: validate a few saved quintet pickles
import random

import numpy as np
from research_core.classes.calibrate import HawkesCalibration

idx_path = DATA_DIR / f"hawkes_triple_quintet_index_{asset}.pkl"
if not idx_path.is_file():
    print("No index file yet. Run the per-quintet cell first.")
else:
    meta = HawkesCalibration.load_params(idx_path)
    ok = [e for e in meta["entries"] if e.get("path")]
    if not ok:
        print("Index has no successful entries.")
    else:
        sample = random.sample(ok, min(3, len(ok)))
        for e in sample:
            d = HawkesCalibration.load_params(e["path"])
            adj = np.asarray(d["adjacency"])
            dec = np.asarray(d["decays"])
            assert adj.shape == (6, 6, 3), adj.shape
            assert dec.shape == (3,), dec.shape
            assert float(d["branching_ratio"]) < 1.0
            print("OK", e["quintet"], e["path"])


## Quintet-kernel simulations (BBO + MO, 1M events)

For each successful row in `hawkes_triple_quintet_index_{asset}.pkl`, workers run `run_quintet_bbo_sim`: **triple** Hawkes from `hawkes_triple_quintet_{asset}_{b:03d}.pkl`, then `load_real_orderbook_snapshot` (as in `notebooks/simulation.ipynb`) on the **first** day in `data/calibration_quintets.json` for that quintet at `10:00:00`, then **1,000,000** events with `recording_mode="medium"` (`bbo` + `mo_orders`). Requires WSE HDF5 `data/WSELOB-2017/orders/{asset}_lob_2017_zlib.h5`.

Databases: `DATA_DIR / "quintet_bbo_sims" / {asset} / bbo_{asset}_{b:03d}.sqlite`. Manifest: `quintet_bbo_sim_manifest_{asset}.pkl` (`db_paths`: quintet index → absolute path).

Uses `ProcessPoolExecutor` with **`QUINTET_SIM_WORKERS = 12`**. Requires prior cells with **`asset`**, **`DATA_DIR`**, completed quintet Hawkes calibration, and WSE HDF5 order files.

After the run, use **`load_quintet_bbo_df(b)`** to load the full BBO series for kernel index `b` (0–99).


In [ ]:
import sqlite3
import traceback
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

import pandas as pd

from research_core.classes.calibrate import HawkesCalibration
from research_core.classes.quintet_sim_bbo import run_quintet_bbo_sim

In [ ]:
import sqlite3
import traceback
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

import pandas as pd

from research_core.classes.calibrate import HawkesCalibration
from research_core.classes.quintet_sim_bbo import run_quintet_bbo_sim

QUINTET_SIM_N_EVENTS = 1_000_000
QUINTET_SIM_WORKERS = 12
QUINTET_SIM_SKIP_EXISTING = False
QUINTET_SIM_START = 0
QUINTET_SIM_END = 100
# BBO + mo_orders (required for stylized facts). Passed explicitly to pool workers.
QUINTET_SIM_RECORDING_MODE = "medium"

OUT_ROOT = DATA_DIR / "quintet_bbo_sims" / str(asset)
OUT_ROOT.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH = DATA_DIR / f"quintet_bbo_sim_manifest_{asset}.pkl"

idx_path = DATA_DIR / f"hawkes_triple_quintet_index_{asset}.pkl"
if not idx_path.is_file():
    raise FileNotFoundError(f"Missing quintet index: {idx_path}")

index_meta = HawkesCalibration.load_params(idx_path)
params_by_quintet = {}
for e in index_meta["entries"]:
    pth = e.get("path")
    if pth and Path(pth).is_file():
        params_by_quintet[int(e["quintet"])] = str(Path(pth).resolve())

jobs = []
for b in range(QUINTET_SIM_START, min(QUINTET_SIM_END, 100)):
    pk = params_by_quintet.get(b)
    if pk is None:
        print(f"[{b:03d}] skip (no calibrated pickle)")
        continue
    db_path = OUT_ROOT / f"bbo_{asset}_{b:03d}.sqlite"
    jobs.append(
        {
            "quintet_index": b,
            "params_path": pk,
            "out_db_path": str(db_path.resolve()),
            "n_events": QUINTET_SIM_N_EVENTS,
            "skip_if_exists": QUINTET_SIM_SKIP_EXISTING,
            "recording_mode": QUINTET_SIM_RECORDING_MODE,
        }
    )

results = []
errors = {}
with ProcessPoolExecutor(max_workers=QUINTET_SIM_WORKERS) as pool:
    futures = {
        pool.submit(
            run_quintet_bbo_sim,
            j["quintet_index"],
            j["params_path"],
            j["out_db_path"],
            n_events=j["n_events"],
            skip_if_exists=j["skip_if_exists"],
            recording_mode=j["recording_mode"],
        ): j["quintet_index"]
        for j in jobs
    }
    for fut in as_completed(futures):
        b = futures[fut]
        try:
            r = fut.result()
            results.append(r)
            tag = "skip" if r.get("skipped") else "ok"
            print(f"[{b:03d}] {tag}  {r.get('db_path', '')}")
        except Exception as e:
            errors[b] = repr(e)
            print(f"[{b:03d}] ERROR {e!r}")
            traceback.print_exc()

manifest = {
    "asset": asset,
    "n_events": QUINTET_SIM_N_EVENTS,
    "out_root": str(OUT_ROOT.resolve()),
    "workers": QUINTET_SIM_WORKERS,
    "recording_mode": QUINTET_SIM_RECORDING_MODE,
    "db_paths": {
        int(r["quintet_index"]): r["db_path"]
        for r in results
        if r.get("db_path")
    },
    "errors": errors,
}
HawkesCalibration.save_params(MANIFEST_PATH, **manifest)
print("Wrote manifest:", MANIFEST_PATH)
print(f"Finished {len(results)} worker result(s), {len(errors)} error(s).")


def load_quintet_bbo_df(quintet_index: int, manifest=None):
    # Load full BBO series for one kernel index from its SQLite file.
    if manifest is None:
        manifest = HawkesCalibration.load_params(MANIFEST_PATH)
    dbp = manifest["db_paths"][int(quintet_index)]
    con = sqlite3.connect(dbp)
    try:
        return pd.read_sql_query(
            "SELECT timestamp, best_bid, best_ask, mid_price FROM bbo ORDER BY rowid",
            con,
        )
    finally:
        con.close()


## Quintet sims: stylized-fact plots (5 groups)

Diagnostic figures from **`AnalyseMarket`** on **five** quintet SQLite files listed in `quintet_bbo_sim_manifest_{asset}.pkl` under `DATA_DIR`. Each database must include **`bbo`** and **`mo_orders`** (e.g. sims run with `recording_mode="medium"`).

Edit **`QUINTET_PLOT_IDS`** to choose which quintet indices to plot. Many panels are produced per quintet in the cell output below.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from research_core.classes import AnalyseMarket
from research_core.classes.calibrate import HawkesCalibration

# Quintet indices to visualize (exactly five by default)
QUINTET_PLOT_IDS = range(0, 99)
MANIFEST_PLOT = DATA_DIR / f"quintet_bbo_sim_manifest_{asset}.pkl"

if not MANIFEST_PLOT.is_file():
    raise FileNotFoundError(f"Missing manifest: {MANIFEST_PLOT}")

mplot = HawkesCalibration.load_params(MANIFEST_PLOT)
paths = mplot.get("db_paths") or {}

for b in QUINTET_PLOT_IDS:
    dbp = paths.get(b)
    if not dbp:
        print(f"[{b:03d}] skip: not in manifest db_paths")
        continue
    print("=" * 72)
    print(f"Quintet {b:03d}")
    print(dbp)

    am = AnalyseMarket(dbp, tick_size=TICK_SIZE, verbose=False)
    if "mo_orders" not in am.tables:
        print("  skip: no mo_orders (need medium recording sim)")
        continue
    if "bbo" not in am.tables:
        print("  skip: no bbo")
        continue

    # 1) Order-sign ACF (MO sides): long-memory stylized fact
    am.stylized_order_sign_autocorrelation()

    # 2) Volatility clustering: |r| and r² ACFs (1 min log returns)
    am.stylized_volatility_clustering(interval_minutes=1, max_lag=100)

    # 3) Return ACF: microstructure vs efficient window (pipeline default)
    am.stylized_return_autocorrelation(interval_minutes=1, max_lag=60)

    # 4) Heavy tails: log price changes vs Gaussian (30 min grid)
    am.stylized_fat_tails(interval_minutes=30, changes="log")

    # 5) Drift: demeaned mid path on a coarse grid + OLS trend
    iv_sec = 5 * 60.0
    sampled = am._get_sampled_mid_prices(iv_sec)
    if sampled is None or len(sampled) < 30:
        print("  drift plot: not enough sampled mids")
    else:
        t = np.linspace(0.0, 1.0, len(sampled))
        m = np.asarray(sampled, dtype=float)
        z = (m - np.nanmean(m)) / (np.nanstd(m) + 1e-12)
        slope, intercept = np.polyfit(t, z, 1)
        fig, ax = plt.subplots(figsize=(10, 3.2))
        ax.plot(t, z, lw=0.85, color="steelblue", label="z(mid)")
        ax.plot(t, slope * t + intercept, ls="--", color="crimson", lw=1.2,
                label=f"OLS slope={slope:.4f}")
        ax.set_title(
            f"Drift check: demeaned mid (z-score), {iv_sec/60:.0f} min sampling"
        )
        ax.set_xlabel("normalized time")
        ax.set_ylabel("z-score")
        ax.legend(loc="best", fontsize=8)
        plt.tight_layout()
        plt.show()


## Long focus runs (5M events): quintets 44, 60, 65

Each cell below runs **`run_quintet_bbo_sim`** with **5,000,000** events (same setup as the 1M batch: triple Hawkes pickle, `load_real_orderbook_snapshot` on the quintet's first day at `10:00:00`, `recording_mode="medium"`).

Output SQLite: `DATA_DIR/quintet_bbo_sims/{asset}/bbo_{asset}_{idx:03d}_5m.sqlite` (does not overwrite the 1M `bbo_{asset}_{idx:03d}.sqlite`).

After each run, the same **stylized-fact plots** as the quintet plot section are shown. A small manifest is merged into **`quintet_bbo_sim_manifest_{asset}_5m_focus.pkl`**.

The helper cell uses **SimulateFast** (classes/simulate_fast.py, Numba) when **QUINTET_5M_USE_SIMULATE_FAST** is True (default). Set it to False to use the reference **Simulate** path.

Run **one cell at a time** (each sim is long). Toggle **`QUINTET_5M_SKIP_IF_EXISTS`** in the helper cell to `True` to skip completed DBs.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from research_core.classes import AnalyseMarket
from research_core.classes.calibrate import HawkesCalibration
from research_core.classes.quintet_sim_bbo import run_quintet_bbo_sim

QUINTET_5M_N_EVENTS = 5_000_000
QUINTET_5M_SKIP_IF_EXISTS = False  # set True to skip if *_5m.sqlite already finished
QUINTET_5M_FLUSH_EVERY = 50_000
QUINTET_5M_RECORDING_MODE = "medium"
QUINTET_5M_USE_SIMULATE_FAST = False  # SimulateFast (Numba) via run_quintet_bbo_sim

OUT_ROOT_5M = DATA_DIR / "quintet_bbo_sims" / str(asset)
OUT_ROOT_5M.mkdir(parents=True, exist_ok=True)
MANIFEST_5M_FOCUS = DATA_DIR / f"quintet_bbo_sim_manifest_{asset}_5m_focus.pkl"


def quintet_focus_5m_run_and_plots(quintet_index: int) -> None:
    """5M-event BBO sim + same stylized diagnostics as the quintet plot cell."""
    idx_path = DATA_DIR / f"hawkes_triple_quintet_index_{asset}.pkl"
    if not idx_path.is_file():
        raise FileNotFoundError(f"Missing quintet index: {idx_path}")

    index_meta = HawkesCalibration.load_params(idx_path)
    pk = None
    for e in index_meta.get("entries", []):
        if int(e.get("quintet", -1)) == int(quintet_index):
            pth = e.get("path")
            if pth and Path(pth).is_file():
                pk = str(Path(pth).resolve())
            break
    if not pk:
        raise FileNotFoundError(f"No calibrated pickle for quintet {quintet_index}")

    db_path = OUT_ROOT_5M / f"bbo_{asset}_{int(quintet_index):03d}_5m.sqlite"
    print("=" * 72)
    print(f"Quintet {int(quintet_index):03d}  |  n_events={QUINTET_5M_N_EVENTS:,}")
    print("out:", db_path)

    r = run_quintet_bbo_sim(
        int(quintet_index),
        pk,
        str(db_path.resolve()),
        n_events=QUINTET_5M_N_EVENTS,
        skip_if_exists=QUINTET_5M_SKIP_IF_EXISTS,
        recording_mode=QUINTET_5M_RECORDING_MODE,
        flush_every=QUINTET_5M_FLUSH_EVERY,
        use_simulate_fast=QUINTET_5M_USE_SIMULATE_FAST,
    )
    if r.get("skipped"):
        print("  (skipped: DB already exists and skip_if_exists=True)")
    dbp = str(db_path.resolve())

    if MANIFEST_5M_FOCUS.is_file():
        man = HawkesCalibration.load_params(MANIFEST_5M_FOCUS)
        db_paths = dict(man.get("db_paths") or {})
    else:
        db_paths = {}
    db_paths[int(quintet_index)] = dbp
    HawkesCalibration.save_params(
        MANIFEST_5M_FOCUS,
        asset=asset,
        n_events=QUINTET_5M_N_EVENTS,
        out_root=str(OUT_ROOT_5M.resolve()),
        recording_mode=QUINTET_5M_RECORDING_MODE,
        db_paths=db_paths,
        errors={},
    )
    print("Wrote / updated manifest:", MANIFEST_5M_FOCUS)

    am = AnalyseMarket(dbp, tick_size=TICK_SIZE, verbose=False)
    if "mo_orders" not in am.tables:
        print("  skip plots: no mo_orders")
        return
    if "bbo" not in am.tables:
        print("  skip plots: no bbo")
        return

    am.stylized_order_sign_autocorrelation()
    am.stylized_volatility_clustering(interval_minutes=1, max_lag=100)
    am.stylized_return_autocorrelation(interval_minutes=1, max_lag=60)
    am.stylized_fat_tails(interval_minutes=30, changes="log")

    iv_sec = 5 * 60.0
    sampled = am._get_sampled_mid_prices(iv_sec)
    if sampled is None or len(sampled) < 30:
        print("  drift plot: not enough sampled mids")
    else:
        t = np.linspace(0.0, 1.0, len(sampled))
        m = np.asarray(sampled, dtype=float)
        z = (m - np.nanmean(m)) / (np.nanstd(m) + 1e-12)
        slope, intercept = np.polyfit(t, z, 1)
        fig, ax = plt.subplots(figsize=(10, 3.2))
        ax.plot(t, z, lw=0.85, color="steelblue", label="z(mid)")
        ax.plot(
            t,
            slope * t + intercept,
            ls="--",
            color="crimson",
            lw=1.2,
            label=f"OLS slope={slope:.4f}",
        )
        ax.set_title(
            f"Drift check: demeaned mid (z-score), {iv_sec / 60:.0f} min sampling (5M run)"
        )
        ax.set_xlabel("normalized time")
        ax.set_ylabel("z-score")
        ax.legend(loc="best", fontsize=8)
        plt.tight_layout()
        plt.show()



In [ ]:
quintet_focus_5m_run_and_plots(44)


In [ ]:
quintet_focus_5m_run_and_plots(60)


In [ ]:
quintet_focus_5m_run_and_plots(65)


## Quintet sims: stylized-fact scores (all groups)

Computes one row per quintet in `quintet_bbo_sim_manifest_{asset}.pkl` via `score_quintet_manifest` (`research_core.classes.sim_stylized_metrics`): MO sign long-memory (power-law on log–log bins from lag ≥ 10, adaptive upper lag before a white-noise tail, exponential alternative and R² comparison), volatility clustering (|r| / return-variance ACF–based scores), mean |return ACF| beyond 20 minutes, 30-minute log-return excess kurtosis, and drift (OLS slope on z-scored 5-minute mids). Saves **`quintet_stylized_scores_{asset}.csv`** under `DATA_DIR`.

Component scores `score_*` and `composite` are heuristic [0, 1] summaries; use raw columns for your own weighting.


In [ ]:
from research_core.classes.calibrate import HawkesCalibration
from research_core.classes.sim_stylized_metrics import score_quintet_manifest

MANIFEST_SCORE = DATA_DIR / f"quintet_bbo_sim_manifest_{asset}.pkl"
if not MANIFEST_SCORE.is_file():
    raise FileNotFoundError(f"Missing manifest: {MANIFEST_SCORE}")

mscore = HawkesCalibration.load_params(MANIFEST_SCORE)
df_scores = score_quintet_manifest(mscore, TICK_SIZE)
OUT_CSV = DATA_DIR / f"quintet_stylized_scores_{asset}.csv"
df_scores.to_csv(OUT_CSV, index=False)
print(df_scores.head())
print("...", len(df_scores), "rows")
print("wrote", OUT_CSV.resolve())
